In [1]:
# Use the following inside the jupyter terminal in sequence. This is for setting up the CPT VLLM model
"""
pip install uv
uv venv --python 3.12 --seed
source .venv/bin/activate
uv pip install vllm --torch-backend=auto
export TIKTOKEN_RS_CACHE_DIR="/lambda/nfs/NDS1"
vllm serve openai/gpt-oss-120b --gpu-memory-utilization 0.85

vllm serve openai/gpt-oss-120b --gpu-memory-utilization 0.9
vllm serve openai/gpt-oss-120b



vllm serve meta-llama/Llama-3.1-8B-Instruct
huggingface-cli login"
"""

'\npip install uv\nuv venv --python 3.12 --seed\nsource .venv/bin/activate\nuv pip install vllm --torch-backend=auto\nexport TIKTOKEN_RS_CACHE_DIR="/lambda/nfs/NDS1"\nvllm serve openai/gpt-oss-120b --gpu-memory-utilization 0.85\n\nvllm serve openai/gpt-oss-120b --gpu-memory-utilization 0.9\nvllm serve openai/gpt-oss-120b\n\n\n\nvllm serve meta-llama/Llama-3.1-8B-Instruct\nhuggingface-cli login"\n'

In [2]:
!pip install openai
!pip install distro
!pip install openpyxl
!pip install transformers
!pip install accelerate
!pip install faiss-cpu
!pip install tf-keras
!pip install anthropic
!pip install sentence_transformers
!pip install chardet

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [3]:
import os
folderpath = r"/lambda/nfs/NDS1/03_MedicalCoding/Demo_Tenet_charts/CVSM_Combined"
ll = len(os.listdir(folderpath))
print(ll)

3986


In [4]:
import sys
sys.path.append("/home/ubuntu/.local/lib/python3.12/site-packages") 
sys.path.append("/usr/lib/python3/dist-packages")
sys.path.append('/lambda/nfs/NDS1/03_MedicalCoding/Endpoint_SourceCodes')
from openai import OpenAI
import os, chardet, time
import pandas as pd
from detection_fns_surgery_CPU import main_execute_single, main_execute_single_icd, main_execute_single_cpt, main_execute_single_hcpcs
# from DL_MODEL_ICD_CPT_combined import *  # Imports ICD and CPT related modules
from DL_MODEL_ICD_CPT_combined_CPU import * # Imports ICD and CPT related modules (CPU based)
import transformers, torchvision, accelerate, torch, requests, re, warnings, json, time, os, csv, openai
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from mybackend import *

/home/ubuntu/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mybackend loaded


In [5]:
import pandas as pd
pd.set_option('display.max_rows', None)      # show all rows
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.width', None)         # don't wrap columns
pd.set_option('display.max_colwidth', None)  # show full column contents

In [6]:
# Load DL Model   # Require transformers version 4.57.3
class ModelLoader:
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(ModelLoader, cls).__new__(cls)
            cls._instance.load_models()
        return cls._instance
    def load_models(self):
            dl_object = DL_Model_ICD_CPT()
            self.model_ICD, self.LABELS_ICD,self.model_CPT, self.LABELS_CPT, self.tokenizer_common = dl_object.Create_DL_Model()
model_loader = ModelLoader()
print("DL Models loaded successfully")
# DF_All, DFA, DFP, ftmod, ftind, ftindextra, all_trees_unique, tree_df_p, tree_database = finetuned_adapter_loader_helper1()
DF_ALL, DFA, DFP, ftmod, ftind, ftindextra, all_trees_unique, tree_df_p, tree_database, df_modifier = finetuned_adapter_loader_helper1()
# DF_ALL_I, ftmod_i, ftind_i = finetuned_adapter_loader_helper3()
print("Finetuned loaded successfully")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /lambda/nfs/NDS1/03_MedicalCoding/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CPT MODEL LOADED SUCCESSFULLY ON CPU


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /lambda/nfs/NDS1/03_MedicalCoding/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ICD MODEL LOADED SUCCESSFULLY ON CPU
DL Models loaded successfully
ALL DB loaded... Please proceed


Batches: 100%|██████████| 2/2 [00:00<00:00, 169.81it/s]


FTModel_loaded
Finetuned loaded successfully


/lambda/nfs/NDS1/03_MedicalCoding/Endpoint_SourceCodes/mybackend.py:181: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  tree_database.iloc[1:] = tree_database.iloc[1:].applymap(lambda x: x.lower() if isinstance(x, str) else x) # All to uppercase


In [7]:
############ Load the master tenet coding summary ############
mastersummary_path = "/lambda/nfs/NDS1/03_MedicalCoding/Demo_Tenet_charts/CVSM_GROUPED_SUMMARY.xlsx"  # 01_Master Tenet Coding Summary_withPain.xlsx
mastersummarydf = pd.read_excel(mastersummary_path)

## Required background functions

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
import pandas as pd
import re

In [9]:
def acquire_CS_CPT_Genito_Gastro(filename_without_txt,mastersummarydf):
    cs_codes = "NOT FOUND"
    if "_" in filename_without_txt:
        filename_without_txt = filename_without_txt.split("_")[0]
    if filename_without_txt in mastersummarydf['ENCOUNTER NO'].astype(str).values:
        cpt_raw = mastersummarydf.loc[
            mastersummarydf['ENCOUNTER NO'].astype(str) == filename_without_txt,
            'Customer CPT'].values[0]
        cpt_string = str(cpt_raw).strip()
        if cpt_string.lower() in ["nan", "", "none"]:
            filtered_cpts = []
        else:
            cpt_list = [c.strip() for c in cpt_string.split(',')]
            filtered_cpts = [c for c in cpt_list if not c.startswith(('8'))]
        if len(filtered_cpts) > 1:     cs_codes = ", ".join(filtered_cpts)
        elif len(filtered_cpts) == 1:  cs_codes = filtered_cpts[0]
        else:  cs_codes = "NOT FOUND"
    return cs_codes

import re
def spelling_checker_correction(text_inp):
    corrections = {
        "ORAL BARIUM": "SINGLE CONTRAST ORAL BARIUM",
        "FIREFLY TECHNOLOGY": "FIREFLY TECHNOLOGY CHOLANGIOGRAPHY",
        "TAP BLOCKS": "TAP BLOCKS BILATERAL INJECTION",
        "EGD": "EGD Esophagogastroduodenoscopy",
        "ESOPHAGOGASTROJEJUNOSCOPY": "EGD Esophagogastroduodenoscopy",
        "ESWL": "ESWL Extracorporeal Shock Wave Lithotripsy",
        "MASK ANESTHESIA" : "GENERAL ANESTHESIA",
        "NVSendo" : "NVSendo Repair of nasal vestibular stenosis",
        "Septo" : "Septo Septoplasty or submucous resection, with or without cartilage",
        "Turbs" : "Turbs Submucous resection inferior turbinate",
        "Synvisc" : "Synvisc viscosupplementation treatment for osteoarthritis",
        "NM" : "NM Nuclear Medicine",
        "SLT" : "Selective Laser Trabeculoplasty",
        "XR " : "XR Radiological examination",
        "XRAY " : "XRAY Radiological examination",
        "Shaving Biopsy" : "Shaving of epidermal or dermal lesion",
        "Shave Biopsy" : "Shaving of epidermal or dermal lesion",
        "WO W" : "WITHOUT AND WITH CONTRAST",
        "Nasal Endo." : "Nasal Endoscopy",
        "Nasal Endo" : "Nasal Endoscopy",
        "Nasal End." : "Nasal Endoscopy",
        "Nasal End." : "Nasal Endoscopy",
    }

    # Build regex pattern for all keys – case insensitive, whole-word friendly
    pattern = re.compile(
        r'\b(' + "|".join(re.escape(k) for k in corrections.keys()) + r')\b',
        flags=re.IGNORECASE
    )

    def replace(match):
        word = match.group(0).upper()
        return corrections.get(word, match.group(0))

    return pattern.sub(replace, text_inp)


In [10]:
def process_single_file(idx, filename):
    try:
        print(f"[PROCESS] {filename}")
        filename_without_txt = filename.replace(".txt", "").strip()
        cs_codes_cpt = acquire_CS_CPT_Genito_Gastro(filename_without_txt,mastersummarydf) # ACQUIRE CODE SUMMARY
        chart_text = read_chart(input_folder, filename) # READ CHART
        chart_text = spelling_checker_correction(chart_text) # 
        # extracted_main = extract_mainprocedure_heading(chart_text)  # EXTRACT MAIN PROCEDURE HEADLINE FROM CHART        
        extracted_main = extract_mainprocedure_statement(chart_text)  # EXTRACT MAIN PROCEDURE HEADLINE FROM CHART        
        extracted_main = extract_mainprocedure_statement_refining(chart_text, extracted_main) # Assign tags
        extracted_main = extract_mainprocedure_statement_refining2(chart_text, extracted_main) # Reverify the tags
        extracted_main = extract_mainprocedure_statement_refining3(chart_text, extracted_main) # Remove the tags        
        mainprocedures = extract_mainprocedures(extracted_main)
        print(filename, "Main Procedure", extracted_main)

        # Route-1: Frame statements using Tree powered by LLM and get codes # tree_llm_statements, tree_llm_codes, tree_DL_codes are all string format
        relevant_parts, relevant_quantities, tree_stats, tree_llm_statements, tree_llm_codes, tree_DL_codes = tree_and_LLMtechnique(extracted_main,chart_text,mainprocedures,DFP,model_loader, ftmod, ftindextra, ftind)
        print("A", tree_llm_statements, tree_llm_codes, tree_DL_codes)
        
        # # Route-2 (Prompting & Framing statements with Finetuned LLM and getting DL and Model Codes)
        # response_CPT, CPT_stats, Model_CPTs_raw, Model_CPTs_FT, DL_CPTs_raw, DL_CPTs_FT, Model_addons = acquire_CPT_Genito(extracted_main,chart_text,ftmod, ftind, DFP, DFA,model_loader,ftindextra)        
        # print("B", CPT_stats, Model_CPTs_FT, DL_CPTs_FT)
        
        # # Route-3 (Applying the extracted main heading to the Finetuned LLM to get CPT code)
        # response_CPT2, CPT_stats2, Model_CPTs_raw2, Model_CPTs_FT2, DL_CPTs_raw2, DL_CPTs_FT2, Model_addons2 = acquire_CPT_Genito2(mainprocedures, extracted_main, extracted_main, chart_text, ftmod, ftind,
        #                                                                                                                            DFP, DFA, model_loader, ftindextra)
        # print("C", CPT_stats2, Model_CPTs_FT2, DL_CPTs_FT2)

        # Find Matching, Good/Bad of Route-1               
        CPT1_CPT2_match_not_match = choose_final2(tree_llm_statements, tree_llm_codes, tree_DL_codes, DFP,chart_text,model_loader,ftmod)
        print("D", CPT1_CPT2_match_not_match)

        # # Find Matching, Good/Bad of Route-2 
        # CPT3_match_not_match, CPT3_GOOD_BAD_LLM = choose_final2(CPT_stats, Model_CPTs_FT, DL_CPTs_FT, DFP,chart_text,model_loader,ftmod)
                
        addoncodes_list = acquire_addoncode2(tree_llm_statements, ftmod, ftind, DFP, DFA, chart_text)           
        print("E", addoncodes_list)

        # Compute MODIFIERS from TREE_LLM_STATEMENTS
        if tree_llm_statements:
            modifiers = compute_modifiers(tree_llm_statements, df_modifier)
        else:
            modifiers = ""

        
        return {"SR": idx, "FILENAME": filename, "MAIN HEADING": extracted_main, "MAIN PROCEDURES": mainprocedures,
                "TREE_PARTS": relevant_parts, "TREE_QUANTITIES" : relevant_quantities,
                "TREE_LLM_RESPONSE": tree_stats, "TREE_LLM_STATEMENTS": tree_llm_statements, "CPT1 TREE_LLM_MODEL_CODES": tree_llm_codes, "CPT2 TREE_LLM_DL_CODES": tree_DL_codes,
                "CPT1_CPT2_MATCH": CPT1_CPT2_match_not_match, "ADDONCODES": addoncodes_list, "MODIFIERS": modifiers, "CS_CODES": cs_codes_cpt}
    except Exception as e:
        print(f"[ERROR] {filename}: {e}")
        return None

## Required Foreground portions

In [ ]:
import os;  import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import time
start_time = time.time()
# input_folder = r"/lambda/nfs/NDS1/03_MedicalCoding/Demo_Tenet_charts/All_Tenet_Charts/4 Radio"   # 1 Genitourinary 8 Gastroenterology  4 Radio  6 CT
input_folder = r"/lambda/nfs/NDS1/03_MedicalCoding/Demo_Tenet_charts/Surgery_Samples" # CVSM_Combined
output_folder_CPT = r"/lambda/nfs/NDS1/03_MedicalCoding/Demo_Tenet_charts/KK_output/Surgery_sample.xlsx"
START = 0 ; END = 10
df_CPT = pd.DataFrame(columns=["SR","FILENAME","MAIN HEADING","MAIN PROCEDURES",
                               "TREE_PARTS", "TREE_QUANTITIES",
                               "TREE_LLM_RESPONSE","TREE_LLM_STATEMENTS","CPT1 TREE_LLM_MODEL_CODES","CPT2 TREE_LLM_DL_CODES", 
                               "CPT1_CPT2_MATCH", "ADDONCODES", "MODIFIERS", "CS_CODES"])
df_lock = Lock();  progress_lock = Lock()
## LIST & SLICE FILES (DETERMINISTIC ORDER) ## 
files = sorted(os.listdir(input_folder))
txt_files = [f for f in files if f.lower().endswith(".txt")]
# Safe slicing
END = min(END, len(txt_files)) ;  txt_files = txt_files[START:END]  ;   total_files = len(txt_files)
print(f"📂 Processing files index {START} to {END-1}") ;  print(f"📊 Total files in this run: {total_files}")
MAX_WORKERS = 10  ;   processed_count = 0
print(f"🚀 Starting parallel processing with {MAX_WORKERS} workers")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(process_single_file, i, fname)
        for i, fname in enumerate(txt_files)]
    for future in as_completed(futures):
        result = future.result()
        if result is None:
            continue
        with df_lock:
            df_CPT.loc[len(df_CPT)] = result
            df_CPT.to_excel(output_folder_CPT, index=False)
        with progress_lock:
            processed_count += 1
            percent = (processed_count / total_files) * 100
            print(f"📈 Progress: {processed_count}/{total_files} "
                f"({percent:.2f}%)")
        print(f"✅ Completed: {result['FILENAME']}")
print("🎉 Processing completed for selected range.")
stop_time = time.time()
time_elapsed = round((stop_time - start_time),2)
print("Total time taken: ", time_elapsed, " seconds")

📂 Processing files index 0 to 8
📊 Total files in this run: 9
🚀 Starting parallel processing with 10 workers
[PROCESS] file1.txt
[PROCESS] file2.txt
[PROCESS] file3.txt
[PROCESS] file4.txt
[PROCESS] file5.txt
[PROCESS] file6.txt
[PROCESS] file7.txt
[PROCESS] file8.txt
[PROCESS] file9.txt


In [ ]:
result